# ప్రాధాన్యత సభ్య మధ్యవర్తితనంతో హోటల్ బుకింగ్

ఈ నోట్‌బుక్ Microsoft Agent Framework ఉపయోగించి **ఫంక్షన్ ఆధారిత మధ్యవర్తితనం**ను ప్రదర్శిస్తుంది. మేము షరతు ఆధారిత వర్క్‌ఫ్లో ఉదాహరణపై ఆధారపడుతూ ప్రాధాన్యత సభ్యులకు ప్రత్యేక సదుపాయాలు ఇచ్చే మధ్యవర్తితన స్థరాన్ని జోడిస్తున్నాము.

## మీరు నేర్చుకునేది:
1. **ఫంక్షన్-ఆధారిత మధ్యవర్తితనం**: ఫంక్షన్ ఫలితాలను ఆపి మార్చడం
2. **కాంటెక్స్ట్ యాక్సెస్**: అమలు తరువాత `context.result` ను చదవడం మరియు మార్చడం
3. **వ్యవసాయ తర్క అమలు**: ప్రాధాన్యత సభ్యుల లాభాలు
4. **ఫలితాన్ని మార్చడం**: వినియోగదారు స్థితి ఆధారంగా ఫంక్షన్ ఫలితాలను మార్చడం
5. **అటువంటి వర్క్‌ఫ్లో, భిన్న ఫలితాలు**: మధ్యవర్తితనం ఆధారిత ప్రవర్తన మార్పులు

## మధ్యవర్తితనంతో వర్క్‌ఫ్లో معم架న:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## షరతు ఆధారిత వర్క్‌ఫ్లోతో ముఖ్య తేడా:

**మధ్యవర్తితనం లేకపోవడం** (14-conditional-workflow.ipynb):
- పారిస్ వద్ద గదులు లేవు → alternative_agent వైపు మార్గం

**మధ్యవర్తితనంతో** (ఈ నోట్‌బుక్):
- సాధారణ వినియోగదారు + పారిస్ → గదులు లేవు → alternative_agent వైపు మార్గం
- ప్రాధాన్యత వినియోగదారు + పారిస్ → 🌟 మధ్యవర్తితనం ఓవర్‌రైడ్ చేస్తుంది! → లభ్యమవుతుంది → booking_agent వైపు మార్గం

## అప్పటికే కావాల్సినవి:
- Microsoft Agent Framework ఇన్స్టాల్ చేయాలి
- షరతు ఆధారిత వర్క్‌ఫ్లోలను అర్థం చేసుకోవాలి (చూసుకోండి 14-conditional-workflow.ipynb)
- GitHub టోకెన్ లేదా OpenAI API కీ
- మధ్యవర్తితనం నమూనాల ప్రాథమిక అవగాహన


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## దశ 1: నిర్మాణాత్మక అవుట్‌పుట్‌ల కోసం పైడాంటిక్ మోడల్స్ నిర్వచించండి

ఈ మోడల్స్ ఏజెంట్లు తిరిగి ఇచ్చే **స్కీమా**ని నిర్వచిస్తాయి. మిడిల్‌వేర్ అందుబాటుదారిత ఫలితాన్ని మార్చినప్పుడు ట్రాక్ చేయడానికి `priority_override` ఫీల్డ్‌ను జోడించాము.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## దశ 2: ప్రాధాన్యత సభ్యుల డేటాబేస్ నిర్వచించండి

ఈ డెమో కోసం, మనం ప్రాధాన్యత సభ్యత్వ డేటాబేస్‌ను అనుకరించుకుంటాము. ఉత్పాదనలో, ఇది ఒక నిజమైన డేటాబేస్ లేదా APIని ప్రశ్నిస్తుంది.

**ప్రాధాన్యత సభ్యులు:**
- `alice@example.com` - VIP సభ్యుడు
- `bob@example.com` - ప్రీమియం సభ్యుడు  
- `priority_user` - పరీక్షా ఖాతా


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## దశ 3: హోటల్ బుకింగ్ టూల్ సృష్టించండి

షరతు ప్రక్రియలాగే, కానీ ఇప్పుడు ఇది మిడిల్‌వేర్ ద్వారా అడ్డుకోబడుతుంది!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## దశ 4: 🌟 ప్రాధాన్యతా పరిశీలన మిడిల్వేర్ సృష్టించండి (ముఖ్య లక్షణం!)

ఇది ఈ నోట్‌బుక్ యొక్క **ప్రధాన కార్యాచరణ**. మిడిల్వేర్:

1. హోటల్_బుకింగ్ ఫంక్షన్ కాల్‌ను **నిరోధిస్తుంది**
2. `next(context)`ని పిలిచేలా సాధారణంగా ఫంక్షన్‌ను **నిర్వహిస్తుంది**
3. `context.result`లో ఫలితాన్ని **పరిశీలిస్తుంది**
4. వినియోగదారు ప్రాధాన్యత ఉన్నప్పుడు మరియు గదులు లేవంటే ఫలితాన్ని **మారుస్తుంది**
5. మోదిఫై చేసిన ఫలితాన్ని ఏజెంట్‌కి **తిరిగి ఇస్తుంది**

**ప్రధాన నమూనా:**
```python
async def my_middleware(context, next):
    await next(context)  # ఫంక్షన్‌ను అమలు చేయండి
    # ఇప్పుడు context.result ఫంక్షన్ అవుట్పుట్‌ను కలిగి ఉంది
    if some_condition:
        context.result = new_value  # oupperదస్త్రం!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## దశ 5: రూటింగ్ కోసం పరిస్థితి ఫంక్షన్లను నిర్వచించండి

సారూప్యమైన పరిస్థితి ఫంక్షన్లు వద్ద, ఇవి స్థిరీకృత అవుట్‌పుట్‌ను పరిశీలించి రూటింగ్‌ను నిర్ణయిస్తాయి.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## దశ 6: కస్టమ్ డిస్ప్లే ఎగ్జిక్యూటర్ సృష్టించండి

ముందటి విధంగా అదే ఎగ్జిక్యూటర్ - చివరి వర్క్‌ఫ్లో అవుట్‌పుట్‌ని ప్రదర్శిస్తుంది.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## దశ 7: పర్యావరణ వేరియబుల్స్ లోడ్ చేయండి

LLM క్లయింట్ ను కాన్ఫిగర్ చేయండి (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## దశ 8: మిడిల్‌వేర్‌తో AI ఏజెంట్లను సృష్టించండి

**ప్రధాన తేడా:** availability_agentని సృష్టించేటప్పుడు, మనం `middleware` పరామితిని పంపిస్తాము!

ఇదే విధంగా priority_check_middlewareని ఏజెంటు యొక్క ఫంక్షన్ పిలుపు పైప్లైన్‌లో ఇంజెక్ట్ చేస్తాము.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## దశ 9: వర్క్‌ఫ్లోని నిర్మించండి

ముందుకి ఉన్నట్టే వర్క్‌ఫ్లో నిర్మాణం - అందుబాటుపై ఆధారపడి షరతుపూర్వక మార్గదర్శనం.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## దశ 10: పరీక్ష కేసు 1 - ప్యారిస్‌లో సాధారణ వినియోగదారు (ఒవర్రైడ్ లేదు)

సాధారణ వినియోగదారు ప్యారిస్ బుక్ చేయడానికి ప్రయత్నిస్తున్నారు → గదులు లేవు → alternative_agent కి మార్గనిర్దేశం చేస్తుంది


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## దశ 11: పరీక్ష కేసు 2 - 🌟 ప్రాధాన్యత వినియోగదారు పారిస్ లో (ఓవర్‌రైడ్‌తో!)

ఒక ప్రాధాన్యత సభ్యుడు పారిస్ బుక్ చేసుకోడానికి ప్రయత్నిస్తాడు → మొదట గదులు లేదు → 🌟 మిడిల్వేర్ ఓవర్‌రైడ్ చేస్తుంది! → booking_agent కి మార్గదర్శనం చేస్తుంది

**ఇది మిడిల్వేర్ శక్తి యొక్క కీలక ప్రదర్శన!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## దశ 12: పరీక్ష కేసు 3 - స్టాక్హోమ్‌లో ప్రాధాన్యత వినియోగదారు (ఇప్పటికే అందుబాటులో ఉన్నది)

ప్రాధాన్యత వినియోగదారు స్టాక్‌హోమ్ ప్రయత్నించు → గదులు అందుబాటులో ఉన్నాయి → ఏదైనా ఓవర్‌రైడ్ అవసరం లేదు → booking_agent కి మార్గదర్శనం చేస్తుంది

ఇది మిడిల్వేర్ అవసరం ఉన్నప్పుడు మాత్రమే పనిచేస్తుందని చూపిస్తుంది!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## ముఖ్య సూచనాలు మరియు మిడిల్వేర్ సూత్రాలు

### ✅ మీరు నేర్చుకున్నవి:

#### **1. ఫంక్షన్-ఆధారిత మిడిల్వేర్ నమూనా**

మిడిల్వేర్ ఫంక్షన్ కాల్స్‌ను సింపుల్ అసింక్ ఫంక్షన్ ఉపయోగించి ఇంటర్సెప్టు చేస్తుంది:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ఫంక్షన్ అమలు చేయక ముందే
    print("Intercepting...")
    
    # ఫంక్షన్‌ను అమలు చేయండి
    await next(context)
    
    # ఫంక్షన్ అమలు తర్వాత - ఫలితాన్ని పరిశీలించండి
    if context.result:
        # అవసరమైతే ఫలితాన్ని మార్చండి
        context.result = modified_value
```

#### **2. సందర్భ ప్రాప్యత మరియు ఫలితం మళ్లింపు**

- `context.function` - పిలవబడుతున్న ఫంక్షన్ యాక్సెస్ చేయండి
- `context.arguments` - ఫంక్షన్ ఆర్గ్యుమెంట్స్ చదవండి
- `context.kwargs` - అదనపు పారామీటర్లకు యాక్సెస్
- `await next(context)` - ఫంక్షన్ నడుపు
- `context.result` - ఫంక్షన్ అవుట్‌పుట్ చదవడం/మార్చడం

#### **3. వ్యాపార లాజిక్ అమలులో**

మా మిడిల్వేర్ ప్రాధాన్యత గల సభ్య ప్రయోజనాలను అమలు చేస్తుంది:
- **సాధారణ వినియోగదారులు**: మార్పులు లేకుండా, సాధారణ విధానం
- **ప్రాధాన్యత వినియోగదారులు**: "అందుబాటు లేదు" → "అందుబాటులో ఉంది"గా మళ్లింపు
- **శరతుల లాజిక్**: అవసరమైనప్పుడు మాత్రమే మళ్లింపు చేస్తుంది

#### **4. ఒకే విధానం, భిన్న ఫలితాలు**

మిడిల్వేర్ శక్తి:
- ✅ విధానం నిర్మాణం పై ఎటువంటి మార్పు లేదు
- ✅ టూల్ ఫంక్షన్ పై మార్పు లేదు
- ✅ షరతు మార్గదర్శక లాజిక్ పై మార్పు లేదు
- ✅ కేవలం మిడిల్వేర్ → భిన్న ప్రవర్తన!

### 🚀 వాస్తవ ప్రపంచ ఉపయోగాలు:

1. **విఐపీ/ప్రీమియం ఫీచర్స్**
   - ప్రీమియం వినియోగదారులకు రేట్ పరిమితులను మళ్లింపు చేయండి
   - వనరులకు ప్రాధాన్యత యాక్సెస్ ఇవ్వండి
   - ప్రీమియం ఫీచర్లను డైనమిక్‌గా అన్లాక్ చేయండి

2. **ఎ/బి టెస్టింగ్**
   - వినియోగదారులను విభిన్న అమలు దిశలకు రూట్ చేయండి
   - ప్రత్యేక వినియోగదారులతో కొత్త ఫీచర్లను పరీక్షించండి
   - దశలవారీగా ఫీచర్ విడుదలలు

3. **భద్రత & అనుగుణ్యత**
   - ఫంక్షన్ కాల్స్ ఆడిట్ చేయండి
   - సున్నితమైన ఆపరేషన్లను బ్లాక్ చేయండి
   - వ్యాపార నియమాలను అమలు చేయండి

4. **సామర్థ్య ఆప్టిమైజేషన్**
   - ప్రత్యేక వినియోగదారుల కోసం ఫలితాలను క్యాషే చేయండి
   - ఆస్థిత ఆపరేషన్లను వీలైనంత తక్కువ చేయండి
   - డైనమిక్ వనరు కేటాయింపు

5. **లోప నిర్వహణ & రీట్రై**
   - లోపాలను అందంగా పట్టు మరియు నిర్వహణ చేయండి
   - రీట్రై లాజిక్ అమలు చేయండి
   - ప్రత్యామ్నాయ అమలులకుFallback చేయండి

6. **లాగింగ్ & మానిటరింగ్**
   - ఫంక్షన్ అమలు సమయాలను ట్రాక్ చేయండి
   - పారామీటర్లు మరియు ఫలితాలను లాగ్ చేయండి
   - వినియోగ నమూనాలను పర్యవేక్షించండి

### 🔑 డेकोరేటర్ల నుండి ముఖ్య తేడాలు:

| ఫీచర్ | డेकोరేటర్ | మిడిల్వేర్ |
|---------|-----------|------------|
| **వ్యాప్తి** | ఒక్క ఫంక్షన్ | ఏజెంట్ లోని అన్ని ఫంక్షన్లు |
| **లచీనం** | నిర్వచన సమయంలో స్థిరం | రన్‌టైమ్‌లో డైనమిక్ |
| **సందర్భం** | పరిమితమైనది | పూర్తి ఏజెంట్ సందర్భం |
| **రచన** | బహుళ డెకరేటర్లు | మిడిల్వేర్ పైప్‌లైన్ |
| **ఏజెంట్ అవేర్నెస్** | కాదు | అవును (ఏజెంట్ స్థితికి యాక్సెస్) |

### 📚 ఎప్పుడు మిడిల్వేర్ ఉపయోగించాలి:

✅ **మిడిల్వేర్ ఉపయోగించండి, మీరు:**
- వినియోగదారు/సెషన్ స్థితిపై ఆధారపడి ప్రవర్తనను మార్చాలి
- బహుళ ఫంక్షన్లకు లాజిక్ వర్తింపజేయాలి
- ఏజెంట్ స్థాయి సందర్భం అవసరం
- క్రాస్-కటింగ్ సమస్యలను (లాగింగ్, ఆథ్ మొదలైనవి) అమలు చేస్తున్నారు

❌ **మిడిల్వేర్ ఉపయోగించవద్దు, మీరు:**
- సింపుల్ ఇన్‌పుట్ ధృవీకరణ (పైడాంటిక్ ఉపయోగించండి)
- ఫంక్షన్-ప్రత్యేక లాజిక్ (ఫంక్షన్‌లో ఉంచండి)
- ఒకసారి మార్పులు (పరివర్తించాల్సిన ఫంక్షన్)

### 🎓 అధునాతన నమూనాలు:

```python
# బహుళ మిడిల్వేర్ (నిర్వహణ క్రమం కీలకం!)
middleware=[
    logging_middleware,      # మొదట లాగ్స్
    auth_middleware,         # ఆపై ఆథ్ తనిఖీ చేస్తుంది
    cache_middleware,        # ఆపై క్యాచ్ తనిఖీ చేస్తుంది
    rate_limit_middleware,   # ఆపై రేట్ లిమిట్స్
    priority_check_middleware  # చివరికి ప్రాధాన్యత తనిఖీ
]

# షరతుల మేర మిడిల్వేర్ అమల�యం
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ఫలితాన్ని మార్చండి
    else:
        # అమలును పూర్తిగా దాటవేయండి
        context.result = cached_value
```

### 🔗 సంబంధించిన సూత్రాలు:

- **ఏజెంట్ మిడిల్వేర్**: agent.run() కాల్స్‌ను ఇంటర్సెప్టు చేస్తుంది
- **ఫంక్షన్ మిడిల్వేర్**: టూల్ ఫంక్షన్ కాల్స్ ఇంటర్సెప్టు చేస్తుంది (మనం ఉపయోగించినది!)
- **మిడిల్వేర్ పైప్‌లైన్**: క్రమానికి అనుగుణంగా అమలయ్యే మిడిల్వేర్ శ్రేణి
- **సందర్భ పంపిణీ**: మిడిల్వేర్ శ్రేణి ద్వారా స్థితిని పంపడం


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
